# RSNA Knee: study viewer
**Edit → Run all**, then pick a study. The dropdown lists the 58 gold studies first, with their positive labels, then report-only studies.
The slider moves through each series at the same relative depth. The titles show plane, fluid-sensitive or not, `SeriesDescription`, slice index and the `Laterality` tag.
Slices are sorted by position along the slice normal, the modality LUT is applied, and the display window is the 1st–99.5th percentile per series. **No flips.**

Report text stays inside Kaggle. Don't paste it into external tools.

In [3]:
import os
from functools import lru_cache
from pathlib import Path

import ipywidgets as W
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pydicom
from IPython.display import clear_output, display
from pydicom.pixels import apply_modality_lut

SPLIT = os.environ.get("VIEWER_SPLIT", "train")
ROOT = next(p for p in [Path(os.environ.get("VIEWER_ROOT", "/nonexistent")),
                        Path("/kaggle/input/competitions/rsna-knee-abnormality-detection"),
                        Path("/kaggle/input/rsna-knee-abnormality-detection")] if p.exists())
series = pd.read_csv(ROOT / f"{SPLIT}_series.csv")
train = pd.read_csv(ROOT / f"{SPLIT}.csv").set_index("StudyInstanceUID")
LABELS = [c for c in train.columns if c not in ("Report",)]
LABELS = [c for c in LABELS if c in train.columns and train[c].dtype != object]
gold = train.dropna(subset=LABELS) if LABELS else train.iloc[:0]
print(ROOT, "| studies:", len(train), "| gold:", len(gold))

StopIteration: 

In [ ]:
@lru_cache(maxsize=24)
def load_series(study, ser):
    """-> (list of float32 slices sorted along the slice normal, info dict)."""
    files = sorted((ROOT / f"{SPLIT}_series" / study / ser).glob("*.dcm"))
    dss = [pydicom.dcmread(f) for f in files]
    if len(dss) == 1 and int(getattr(dss[0], "NumberOfFrames", 1) or 1) > 1:  # multi-frame file
        vol = list(apply_modality_lut(dss[0].pixel_array, dss[0]).astype(np.float32))
    else:
        iop = np.array(dss[0].get("ImageOrientationPatient", [1, 0, 0, 0, 1, 0]), float)
        normal = np.cross(iop[:3], iop[3:])
        key = [float(normal @ np.array(d.ImagePositionPatient, float)) if "ImagePositionPatient" in d
               else float(d.get("InstanceNumber", 0)) for d in dss]
        vol = [apply_modality_lut(dss[i].pixel_array, dss[i]).astype(np.float32) for i in np.argsort(key)]
    sample = np.concatenate([v[::4, ::4].ravel() for v in vol])
    d = dss[0]
    info = {"window": np.percentile(sample, [1, 99.5]), "desc": str(d.get("SeriesDescription", "?")),
            "lat": str(d.get("Laterality", "-")), "multiframe": len(dss) == 1 and len(vol) > 1,
            "spacing": d.get("PixelSpacing", None)}
    return vol, info


def show(study, pos=0.5):
    rows = series[series.StudyInstanceUID == study].sort_values(["Anatomical_Plane", "Fluid_Sensitive"])
    fig, axes = plt.subplots(1, len(rows), figsize=(3.6 * len(rows), 4.2))
    for ax, r in zip(np.atleast_1d(axes), rows.itertuples()):
        vol, info = load_series(study, r.SeriesInstanceUID)
        i = min(int(pos * len(vol)), len(vol) - 1)
        ax.imshow(vol[i], cmap="gray", vmin=info["window"][0], vmax=info["window"][1])
        ax.set_title(f"{r.Anatomical_Plane[:3]} {'FS' if r.Fluid_Sensitive else 'non-FS'} | {info['desc'][:22]}\n"
                     f"slice {i + 1}/{len(vol)}  lat={info['lat']}{'  (multi-frame)' if info['multiframe'] else ''}",
                     fontsize=8)
        ax.axis("off")
    plt.tight_layout()
    plt.show()


def positives(label):
    """Gold studies positive for `label`, e.g. positives('ACL')."""
    return gold.index[gold[label] == 1].tolist()

In [ ]:
if os.environ.get("KAGGLE_KERNEL_RUN_TYPE") == "Batch":
    # saved (batch) runs have no browser for widgets, which hang; render one static example instead
    u = gold.index[0]
    print("gold labels:", {l: int(gold.at[u, l]) for l in LABELS})
    show(u, 0.5)
else:
    def _name(u):
        if u in gold.index:
            pos = [l for l in LABELS if gold.at[u, l] == 1]
            return f"GOLD | {', '.join(pos) or 'no findings'} | …{u[-8:]}"
        return f"report-only | …{u[-8:]}"

    others = [u for u in train.index if u not in gold.index]
    study_w = W.Dropdown(options=[(_name(u), u) for u in list(gold.index) + others[:500]],
                         description="study", layout=W.Layout(width="95%"))
    pos_w = W.FloatSlider(value=0.5, min=0, max=1, step=0.02, description="depth", continuous_update=False,
                          layout=W.Layout(width="60%"))
    report_w = W.Checkbox(value=False, description="show report")
    out = W.Output()

    def update(*_):
        with out:
            clear_output(wait=True)
            u = study_w.value
            if u in gold.index:
                print("gold labels:", {l: int(gold.at[u, l]) for l in LABELS})
            if report_w.value and "Report" in train.columns:
                print(train.at[u, "Report"])
            show(u, pos_w.value)

    for w in (study_w, pos_w, report_w):
        w.observe(update, "value")
    display(W.VBox([study_w, pos_w, report_w, out]))
    update()

Any study by ID: `show('<StudyInstanceUID>', pos=0.5)`. Gold positives for a label: `positives('Lateral Meniscus')`.